# Notebook 08: The Command Primitive, Functional API & Long-Term Memory
## LangGraph 1.1.9 — Advanced Patterns

This notebook covers three major additions to LangGraph post-1.0.5:

1. **The `Command` Class** — Combine state updates and routing in a single return value
2. **The Functional API** — `@entrypoint` and `@task` decorators as an alternative to `StateGraph`
3. **Long-Term Memory with Stores** — Cross-thread, cross-session memory

**Prerequisites**: Notebooks 01-07  
**Audience**: Advanced — assumes fluency with StateGraph, agents, checkpointers  
**Time**: 60-75 minutes  
**LangGraph Version**: >=1.1.9

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify versions
import langgraph
import langchain

# Check API key
if not os.getenv("NVIDIA_API_KEY"):
    raise ValueError("NVIDIA_API_KEY not set. Add it to your .env file.")
print("\u2705 API key configured")

✅ API key configured


In [2]:
import time

def _invoke_with_backoff(runnable, input_data, config=None, max_retries=5):
    """Invoke any LangChain runnable with exponential backoff on 429 rate-limit errors."""
    delay = 5
    for attempt in range(max_retries):
        try:
            if config:
                return runnable.invoke(input_data, config)
            return runnable.invoke(input_data)
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                print(f"⏳ Rate limited — retrying in {delay}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(delay)
                delay *= 2
            else:
                raise

print("✅ Backoff helper loaded")


✅ Backoff helper loaded


---
# Part 1: The `Command` Class — Combined State Update + Routing

## The Problem Command Solves

In LangGraph, nodes update state and routing functions decide where to go next. Often, the routing decision depends on work already done in the node — meaning you compute something twice:

```python
# Traditional approach — compute in node, then re-compute in routing function
def classify(state) -> dict:
    category = expensive_classify(state["text"])  # <- computed here
    return {"category": category}

def route(state) -> str:
    category = state["category"]                  # <- read result from state
    return f"handle_{category}"

graph.add_conditional_edges("classify", route)
```

**Command eliminates this duplication:**

```python
# Command approach — compute once, route immediately
def classify(state) -> Command:
    category = expensive_classify(state["text"])  # <- computed once
    return Command(
        update={"category": category},             # <- update state
        goto=f"handle_{category}",                 # <- AND route — no separate function!
    )
# No add_conditional_edges needed!
```

In [3]:
from typing_extensions import TypedDict
from typing import Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

class SupportState(TypedDict):
    message: str
    category: str
    priority: str
    response: str

def classify_ticket(state: SupportState) -> Command:
    """Classify support ticket and route — in a single step."""
    message = state["message"].lower()

    # Determine category
    if any(word in message for word in ["billing", "charge", "refund", "payment"]):
        category = "billing"
        priority = "high"
    elif any(word in message for word in ["bug", "error", "crash", "broken"]):
        category = "technical"
        priority = "high"
    elif any(word in message for word in ["feature", "request", "suggest"]):
        category = "feature_request"
        priority = "low"
    else:
        category = "general"
        priority = "medium"

    print(f"\U0001f3f7\ufe0f  Classified: {category} ({priority} priority)")

    return Command(
        update={"category": category, "priority": priority},
        goto=f"handle_{category}",
    )

def handle_billing(state: SupportState) -> dict:
    return {"response": f"\U0001f4b3 Billing team will review: '{state['message'][:50]}...'"}

def handle_technical(state: SupportState) -> dict:
    return {"response": f"\U0001f527 Engineering team assigned: '{state['message'][:50]}...'"}

def handle_feature_request(state: SupportState) -> dict:
    return {"response": f"\U0001f4a1 Added to product backlog: '{state['message'][:50]}...'"}

def handle_general(state: SupportState) -> dict:
    return {"response": f"\U0001f4e7 General support team: '{state['message'][:50]}...'"}

# Build graph — no add_conditional_edges needed!
builder = StateGraph(SupportState)
builder.add_node("classify", classify_ticket)
builder.add_node("handle_billing", handle_billing)
builder.add_node("handle_technical", handle_technical)
builder.add_node("handle_feature_request", handle_feature_request)
builder.add_node("handle_general", handle_general)

builder.add_edge(START, "classify")
for handler in ["handle_billing", "handle_technical", "handle_feature_request", "handle_general"]:
    builder.add_edge(handler, END)

app = builder.compile()

# Test various ticket types
test_tickets = [
    "I was charged twice for my subscription last month",
    "The app crashes when I try to export PDF files",
    "Can you add dark mode to the dashboard?",
    "How do I update my email address?",
]

for ticket in test_tickets:
    result = _invoke_with_backoff(app, {"message": ticket, "category": "", "priority": "", "response": ""})
    time.sleep(5)  # avoid 429 rate-limit
    print(f"\u2192 {result['response']}\n")

/Users/aashishgarg/Downloads/Langraph/langraph-tutorials/.venv/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


🏷️  Classified: billing (high priority)
→ 💳 Billing team will review: 'I was charged twice for my subscription last month...'

🏷️  Classified: technical (high priority)
→ 🔧 Engineering team assigned: 'The app crashes when I try to export PDF files...'

🏷️  Classified: general (medium priority)
→ 📧 General support team: 'Can you add dark mode to the dashboard?...'

🏷️  Classified: general (medium priority)
→ 📧 General support team: 'How do I update my email address?...'



## Command for HITL Resumption

`Command(resume=...)` is the canonical way to resume a paused graph. We saw this in Notebook 05 — here's a more complete picture of all Command variants:

```python
from langgraph.types import Command

# Resume a paused graph
graph.invoke(Command(resume="yes"), config)

# Resume AND update state simultaneously
graph.invoke(Command(resume="approved", update={"reviewer": "alice"}), config)

# Escape from subgraph to parent graph
return Command(goto="parent_node", graph=Command.PARENT, update={"escalated": True})
```

In [4]:
# Complete Command reference with all parameters
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

class ReviewState(TypedDict):
    content: str
    reviewer: str
    approved: bool
    final_content: str

def generate_content(state: ReviewState) -> dict:
    content = f"Generated content based on requirements: {state['content']}"
    print(f"\u270d\ufe0f  Generated: {content}")
    return {"content": content}

def review_gate(state: ReviewState) -> Command:
    """Ask for human review — pause and wait."""
    decision = interrupt({
        "message": f"Review this content:\n\n{state['content']}\n\nApprove?",
        "options": ["approve", "reject", "request_changes"],
    })

    if decision == "approve":
        return Command(
            update={"approved": True},
            goto="publish",
        )
    elif decision == "request_changes":
        return Command(
            update={"approved": False},
            goto="generate_content",  # Loop back!
        )
    else:
        return Command(
            update={"approved": False},
            goto="cancel",
        )

def publish(state: ReviewState) -> dict:
    print(f"\U0001f680 Published: {state['content']}")
    return {"final_content": state["content"]}

def cancel(state: ReviewState) -> dict:
    print("\u274c Content rejected")
    return {"final_content": "REJECTED"}

db = sqlite3.connect(":memory:",check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ReviewState)
builder.add_node("generate_content", generate_content)
builder.add_node("review_gate", review_gate)
builder.add_node("publish", publish)
builder.add_node("cancel", cancel)

builder.add_edge(START, "generate_content")
builder.add_edge("generate_content", "review_gate")
builder.add_edge("publish", END)
builder.add_edge("cancel", END)

app = builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "content-001"}}

# Run to interrupt
print("=== Running to review gate ===")
result = _invoke_with_backoff(app, 
    {"content": "AI safety best practices", "reviewer": "", "approved": False, "final_content": ""},
    config
)
time.sleep(5)  # avoid 429 rate-limit

# Approve the content
print("\n=== Reviewer approves ===")
final = _invoke_with_backoff(app, Command(resume="approve"), config)
print(f"Final: {final['final_content']}")

=== Running to review gate ===
✍️  Generated: Generated content based on requirements: AI safety best practices

=== Reviewer approves ===
🚀 Published: Generated content based on requirements: AI safety best practices
Final: Generated content based on requirements: AI safety best practices


---
# Part 2: The Functional API — @entrypoint and @task

## What Is the Functional API?

LangGraph's `StateGraph` is powerful but requires defining TypedDict schemas, nodes, edges, and calling `.compile()`. The **Functional API** (`@entrypoint` / `@task`) lets you write LangGraph workflows using standard Python code — no explicit graph construction needed.

| Aspect | StateGraph | Functional API |
|---|---|---|
| State management | TypedDict schema | Function parameters |
| Control flow | Nodes + edges | Standard Python (if/while/for) |
| Visualization | Yes (graph diagram) | No |
| Checkpointing | Per-node granularity | Per-workflow granularity |
| Async support | Limited | First-class |
| Interoperability | Works as @task target | Works as StateGraph node |
| Best for | Complex branching, visualization | Sequential workflows, async tasks |

**Both compile to the same runtime** — they interoperate freely.

In [5]:
import asyncio
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import MemorySaver
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

# @task — individual unit of work (lazy evaluation)
@task
def research_topic(topic: str) -> str:
    """Research a topic using the LLM."""
    print(f"\U0001f52c Researching: {topic}")
    response = _invoke_with_backoff(llm, f"Give me 2 key facts about: {topic}. Be concise.")
    time.sleep(5)  # avoid 429 rate-limit
    return response.content

@task
def summarize_findings(findings: list) -> str:
    """Summarize multiple research findings."""
    print(f"\U0001f4ca Summarizing {len(findings)} findings...")
    combined = "\n".join(f"- {f}" for f in findings)
    response = _invoke_with_backoff(llm, f"Summarize these findings in 2 sentences:\n{combined}")
    time.sleep(5)  # avoid 429 rate-limit
    return response.content

# @entrypoint — top-level workflow (has checkpointing, full lifecycle)
@entrypoint(checkpointer=MemorySaver())
def research_workflow(query: str) -> dict:
    """Research multiple related topics and synthesize findings.

    Standard Python control flow — no graph construction needed!
    """
    # Determine sub-topics (could also call another @task here)
    topics = [f"{query} overview", f"{query} applications", f"{query} challenges"]

    # Launch parallel research tasks (lazy — .result() waits for completion)
    research_tasks = [research_topic(topic) for topic in topics]

    # Wait for all results
    findings = [t.result() for t in research_tasks]

    # Summarize
    summary = summarize_findings(findings).result()

    return {
        "query": query,
        "findings": findings,
        "summary": summary
    }

# Call it like a regular function
print("=== Functional API Research Workflow ===")
result = research_workflow.invoke(
    "quantum computing",
    config={"configurable": {"thread_id": "session-1"}}
)
print(f"\nQuery: {result['query']}")
print(f"\nSummary: {result['summary']}")

=== Functional API Research Workflow ===
🔬 Researching: quantum computing overview
🔬 Researching: quantum computing applications
🔬 Researching: quantum computing challenges
📊 Summarizing 3 findings...

Query: quantum computing

Summary: Here's a 2-sentence summary of the findings:

Quantum computing has the potential to revolutionize various fields by solving complex problems in cryptography, optimization, and simulation, leading to breakthroughs in medicine, finance, and climate modeling. However, the development of practical quantum computers is hindered by challenges such as error correction and scalability, which must be addressed to unlock the full potential of quantum computing.


## Interoperability: Functional API and StateGraph

`@entrypoint` workflows and `StateGraph` graphs can call each other. A `@task` can be a StateGraph node, and an `@entrypoint` can invoke a compiled graph.

In [6]:
from langgraph.func import entrypoint, task
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# A @task that wraps a compiled graph
@task
def run_subgraph(input_text: str) -> str:
    """Run a StateGraph as a task inside an entrypoint."""
    # Build a mini StateGraph
    def process(state):
        return {"messages": [{"role": "assistant", "content": f"Processed: {input_text}"}]}

    builder = StateGraph(MessagesState)
    builder.add_node("process", process)
    builder.add_edge(START, "process")
    builder.add_edge("process", END)
    graph = builder.compile()

    result = _invoke_with_backoff(graph, {"messages": [HumanMessage(input_text)]})
    time.sleep(5)  # avoid 429 rate-limit
    return result["messages"][-1].content

@entrypoint(checkpointer=MemorySaver())
def mixed_workflow(inputs: list) -> list:
    """Functional API calling StateGraph as tasks."""
    tasks = [run_subgraph(inp) for inp in inputs]
    return [t.result() for t in tasks]

result = mixed_workflow.invoke(
    ["Hello", "World", "LangGraph"],
    config={"configurable": {"thread_id": "mixed-001"}}
)
for r in result:
    print(r)

Processed: Hello
Processed: World
Processed: LangGraph


---
# Part 3: Long-Term Memory with Stores

## Checkpointer vs Store — The Key Distinction

We've used checkpointers extensively. Stores are fundamentally different:

```
Checkpointer (Short-Term Memory):
  |-- Scope: ONE thread (one conversation)
  |-- Key: thread_id + checkpoint ID
  |-- Lifetime: Until thread deleted
  +-- Use: Conversation state, multi-turn context

Store (Long-Term Memory):
  |-- Scope: ALL threads and sessions
  |-- Key: namespace tuple + item key
  |-- Lifetime: Permanent (until explicitly deleted)
  +-- Use: User preferences, learned facts, cross-session context
```

**Analogy**: Checkpointer = working RAM, Store = hard drive.

In [7]:
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage

db = sqlite3.connect(":memory:",check_same_thread=False)
checkpointer = SqliteSaver(db)
store = InMemoryStore()

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

def smart_assistant(state: MessagesState, store: BaseStore) -> dict:
    """Assistant that learns and remembers user preferences across conversations."""
    # Get user identity (in production, from auth context or config)
    user_id = "alice"

    # Read long-term memories
    user_memories = store.search(("users", user_id, "memories"))

    # Build context from memories
    memory_context = ""
    if user_memories:
        facts = [item.value.get("content", "") for item in user_memories]
        memory_context = "\n\nWhat I know about you:\n" + "\n".join(f"\u2022 {f}" for f in facts)

    # Learn from this message
    last_message = state["messages"][-1].content

    # Extract and store new facts (simplified rule-based extraction)
    facts_to_store = []
    if "i prefer" in last_message.lower() or "i like" in last_message.lower():
        facts_to_store.append(last_message)
    if "my name is" in last_message.lower():
        name = last_message.lower().split("my name is")[-1].strip().split()[0]
        facts_to_store.append(f"User's name is {name}")

    for i, fact in enumerate(facts_to_store):
        key = f"fact_{len(user_memories) + i}"
        store.put(("users", user_id, "memories"), key, {"content": fact})
        print(f"\U0001f4be Stored to long-term memory: {fact}")

    # Generate response with memory context
    system_prompt = f"You are a helpful, personalized assistant.{memory_context}"
    response = llm.invoke(
        [{"role": "system", "content": system_prompt}] + state["messages"]
    )

    return {"messages": [response]}

# Build graph with BOTH checkpointer and store
builder = StateGraph(MessagesState)
builder.add_node("assistant", smart_assistant)
builder.add_edge(START, "assistant")
builder.add_edge("assistant", END)

app = builder.compile(checkpointer=checkpointer, store=store)

# Session 1 — user introduces themselves
print("=== Session 1 ===")
config1 = {"configurable": {"thread_id": "chat-001"}}
r = _invoke_with_backoff(app, {"messages": [HumanMessage("Hi! My name is Alice. I prefer Python over JavaScript.")]}, config1)
time.sleep(5)  # avoid 429 rate-limit
print("Bot:", r["messages"][-1].content)

# Session 2 — DIFFERENT thread, but store remembers Alice
print("\n=== Session 2 (new thread — store still remembers!) ===")
config2 = {"configurable": {"thread_id": "chat-002"}}
r = _invoke_with_backoff(app, {"messages": [HumanMessage("Hey, what do you know about me?")]}, config2)
print("Bot:", r["messages"][-1].content)

=== Session 1 ===
💾 Stored to long-term memory: Hi! My name is Alice. I prefer Python over JavaScript.
💾 Stored to long-term memory: User's name is alice.
Bot: Hello Alice. It's nice to meet you. I'm happy to assist you with any Python-related questions or tasks you may have. What can I help you with today? Do you have a specific project in mind or are you looking for general guidance on Python programming?

=== Session 2 (new thread — store still remembers!) ===
Bot: Nice to meet you, alice. I know a few things about you. You prefer Python over JavaScript, and your name is Alice. Is there anything else you'd like to share or any specific topics you'd like to discuss? I'm here to help.


## Store Namespaces

Stores use a **namespace tuple** to organize data. Think of namespaces as folder paths:

```python
("users", user_id, "preferences")          # user preferences
("users", user_id, "conversation_history") # past conversations
("products", product_id, "reviews")        # product reviews
("global", "configuration")               # app-wide settings
```

You can search within a namespace with semantic queries (when using a vector-enabled store like `PostgresStore`).

In [8]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Store user preferences in organized namespaces
store.put(("users", "alice", "preferences"), "theme", {"value": "dark", "set_at": "2026-04-25"})
store.put(("users", "alice", "preferences"), "language", {"value": "Python", "set_at": "2026-04-25"})
store.put(("users", "alice", "skills"), "python", {"level": "expert", "years": 5})
store.put(("users", "alice", "skills"), "javascript", {"level": "intermediate", "years": 2})

store.put(("users", "bob", "preferences"), "theme", {"value": "light"})
store.put(("users", "bob", "skills"), "java", {"level": "expert"})

# Search within a namespace
alice_prefs = store.search(("users", "alice", "preferences"))
print("Alice's preferences:")
for item in alice_prefs:
    print(f"  {item.key}: {item.value}")

# Get a specific item
alice_theme = store.get(("users", "alice", "preferences"), "theme")
print(f"\nAlice's theme: {alice_theme.value['value']}")

# Delete an item
store.delete(("users", "alice", "preferences"), "language")
print(f"\nAfter deletion, Alice has {len(store.search(('users', 'alice', 'preferences')))} preferences")

# Cross-user operation — get all users' themes
for user in ["alice", "bob"]:
    pref = store.get(("users", user, "preferences"), "theme")
    if pref:
        print(f"{user}'s theme: {pref.value['value']}")

Alice's preferences:
  theme: {'value': 'dark', 'set_at': '2026-04-25'}
  language: {'value': 'Python', 'set_at': '2026-04-25'}

Alice's theme: dark

After deletion, Alice has 1 preferences
alice's theme: dark
bob's theme: light


---
## Exercises

### Exercise 1: Convert to Command
Take this traditional conditional routing code and rewrite it using `Command`:

```python
# Traditional approach — convert this to Command
def analyze_sentiment(state) -> dict:
    text = state["text"]
    # Simplified sentiment
    if "great" in text or "love" in text:
        sentiment = "positive"
    elif "bad" in text or "hate" in text:
        sentiment = "negative"
    else:
        sentiment = "neutral"
    return {"sentiment": sentiment}

def route_sentiment(state) -> str:
    return f"handle_{state['sentiment']}"

graph.add_conditional_edges("analyze", route_sentiment)
```

### Exercise 2: Functional API Research Pipeline
Build a research pipeline using `@entrypoint` and `@task` that:
1. Takes a research question as input
2. Breaks it into 3 sub-questions (@task)
3. Researches each sub-question in parallel (@task, using LLM)
4. Synthesizes the findings (@task)
5. Returns the synthesis

### Exercise 3: Personalized Shopping Assistant
Build a shopping assistant using InMemoryStore that:
1. Remembers user budget preferences across sessions
2. Remembers previously liked/disliked product categories
3. Uses this memory to make personalized recommendations
4. Updates memory when user expresses new preferences

<details>
<summary>Exercise 1 Solution</summary>

```python
from langgraph.types import Command

def analyze_and_route_sentiment(state) -> Command:
    text = state["text"]
    if "great" in text or "love" in text:
        sentiment = "positive"
    elif "bad" in text or "hate" in text:
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return Command(
        update={"sentiment": sentiment},
        goto=f"handle_{sentiment}",
    )
# No add_conditional_edges needed!
```
</details>

---
## Summary

**The `Command` Class** eliminates the need to write separate routing functions when the routing logic depends on work already done in the node. It's concise and explicit.

**The Functional API** (`@entrypoint`/`@task`) provides a Python-native way to write LangGraph workflows without explicit graph construction. Ideal for async-heavy, sequential, or script-like workflows. Fully interoperable with StateGraph.

**Stores** provide cross-thread, cross-session long-term memory — the missing piece for truly personalized agents. Use `InMemoryStore` for development, `PostgresStore` for production.

**Key decisions:**
- Use `Command` when routing depends on expensive computation in the node
- Use Functional API for sequential workflows, scripts, or when visualization is not needed
- Use Stores for user preferences, learned facts, and multi-session continuity
- Keep using StateGraph for complex multi-branch workflows that benefit from graph visualization

## Next Steps
**Notebook 09**: Multi-Agent Systems — Supervisor, Swarm, and custom architectures for coordinating multiple specialized agents